<div style="background:linear-gradient(120deg,#07062e 0%,#120c5c 60%,#2a1a9e 100%);border-radius:14px;padding:26px 30px;color:#ffffff;font-family:'Segoe UI',Arial,sans-serif;"><div style="font-size:12px;letter-spacing:3px;color:#4de1ff;font-weight:600;">MICROSOFT IGNITE 2026 &nbsp;&middot;&nbsp; LAB663</div><div style="font-size:30px;font-weight:700;margin:8px 0 4px 0;color:#ffffff;">Module 6 &middot; Make it optimal</div><div style="font-size:15px;color:#cfd4ff;">The hill climb &middot; about 14 minutes</div><div style="margin-top:16px;"><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">1</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">2</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">3</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">4</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#3cf26b;color:#07062e;">5</span><span style="display:inline-block;width:26px;height:26px;line-height:26px;text-align:center;border-radius:13px;font-size:13px;font-weight:700;margin-right:6px;background:#4de1ff;color:#07062e;">6</span></div><div style="height:6px;border-radius:3px;margin-top:18px;background:linear-gradient(90deg,#3cf26b,#4de1ff,#6c4dff,#ff4fd8,#ff9f1c,#ffd23f);"></div></div>

# 6. Make it optimal (v2 to v3)

**Time:** about 14 minutes. The last climb.

**The question we're starting with:** can I cut my model bill without losing quality?

Quality is up. But look at what we're doing: **every question, even "how much does it weigh?", goes to our biggest model.** That's like taking a helicopter to the corner shop.

This time we change the **model**, not the instructions. We'll swap our single model for a **model router**: one endpoint that picks a model for each request. Then we'll ask the same hard question we always ask:

> **Did quality hold, and did the cost go down?** And we'll answer with the same rubric.

We deployed **four router editions**, so we get to compare them:

| Deployment | How it's set up |
|---|---|
| `mr-default` | The router's own defaults |
| `mr-cost` | Leans toward cheaper models |
| `mr-quality` | Leans toward stronger models |
| `mr-subset` | Only picks from a list we chose |

| Our name | What changes |
|---|---|
| v2 | Optimized instructions (Module 5) |
| v3 | **Today:** same instructions, right-sized model |


---

## 6.1 Where did we leave off?

Module 5 saved a few facts (our rubric, our datasets, our scores) so we don't have to redo them.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
import lab

project_client, openai_client = lab.connect()
state = lab.load_state()
assert "evaluation_id" in state and "v2_version" in state, "Run notebook 05 first so we have a rubric and a v2."

test_cases = lab.load_cases("test")
versions = {int(v.version): v for v in project_client.agents.list_versions(lab.AGENT_NAME)}
v2 = versions[int(state["v2_version"])]
print(f"v2 = Foundry version {v2.version}, model {v2.definition.model}")

## 6.2 Am I overpaying?

If most of our questions are easy, a frontier model on every one is wasted money. Let's count how our test questions break down.

**Predict:** what share of our questions are `easy`?

In [ ]:
mix = Counter(case["difficulty"] for case in test_cases)
print(dict(mix))
print(f"Easy questions: {mix['easy']} of {len(test_cases)}")

**Think:** in a real store, would the mix be more like this or more skewed to easy? Your answer is roughly how much a router could save you.

## 6.3 How does a router decide?

Before we trust a router, let's *watch* one decide. We'll send easy and hard questions to each edition and look at which model answered. The reply tells us.

In [ ]:
probes = {
    "easy": "How much does the TrailLite Daypack weigh?",
    "hard": (
        "A family of six is camping in heavy rain, wants the lightest tent possible, "
        "and has a tight budget. Weigh the trade-offs and recommend one."
    ),
}

rows = []
for edition in lab.ROUTERS:
    for level, question in probes.items():
        reply = openai_client.responses.create(model=edition, input=question)
        # reply.model is the model the router actually used.
        rows.append({"edition": edition, "question": level, "router chose": reply.model})

pd.DataFrame(rows).pivot(index="edition", columns="question", values="router chose")

**Expected:** a small table with four editions down the side, `easy` and `hard` across the top, and a model name in each cell.

**Go deeper:** all four `mr-` deployments are the same router with a different `routing` setting: a mode (`cost`, `balanced` or `quality`) and, for `mr-subset`, a list of allowed models. You can read them in `infra/modules/foundry.bicep`.

**Think:**
- Did `mr-cost` choose differently from `mr-quality`?
- Did any edition send the easy question to a small model and the hard one to a bigger one?
- Does `mr-subset` ever pick a model outside the list we gave it?

This is the thing our hand-built router from Module 3 was trying to do. Here the platform keeps it up to date for us.

## 6.4 How do I test each router edition fairly?

Now the careful part. We keep v2's instructions and the same file search. We change **only the model**, once per edition. That gives us four candidate agents, and any difference between them is because of the router setting.

In [ ]:
from azure.ai.projects.models import FileSearchTool, PromptAgentDefinition

vector_store = next(s for s in openai_client.vector_stores.list() if s.name == "trailmate-manuals")

candidates = {}
for edition in lab.ROUTERS:
    saved = project_client.agents.create_version(
        lab.AGENT_NAME,
        definition=PromptAgentDefinition(
            model=edition,                              # the ONLY thing that changes
            instructions=v2.definition.instructions,    # v2's optimized instructions
            tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
        ),
    )
    candidates[edition] = saved
    print(f"{edition}: Foundry version {saved.version}")

## 6.5 Which edition holds quality for less?

Same rubric, same held-out `test` questions, two repeats each. We watch four things: **score**, **spread** (noise), **tokens** (our cost stand-in) and **seconds**. The `models` column shows which models the router really used.

This is the biggest batch in the lab, so it takes a few minutes. Think about which edition you expect to win while it runs.

In [ ]:
runs = {
    edition: lab.run_repeated_eval(
        openai_client, state["evaluation_id"], state["test_dataset_id"], lab.AGENT_NAME,
        candidates[edition].version, f"v3 candidate {edition}",
    )
    for edition in lab.ROUTERS
}

v2_row = state["test_scores"]["v2 on test"]
board = pd.DataFrame(
    [{"version": "v2 (frontier model)", "score": v2_row["score"], "spread": v2_row["spread"],
      "tokens": v2_row["tokens"], "models": lab.FRONTIER_MODEL}]
    + [{"version": name, "score": r["score"], "spread": r["spread"], "tokens": r["tokens"],
        "models": ", ".join(r["models"])} for name, r in runs.items()]
)
board

**Expected:** five rows: v2 and our four editions, each with a score, a spread, tokens and the models used. This is the longest-running cell in the lab.

**Read it like a detective.**
- Which edition used the **fewest tokens**? Is its score still close to v2's?
- Is any score gap smaller than the spread? If so, we can't tell those two apart.
- Does the `models` column match what you saw in section 3?

If prices are filled in `data/pricing.json`, the next cell turns tokens into dollars. If not, tokens are still a fair stand-in.

## 6.6 Where does the cheap edition slip?

An average hides a lot. Let's break each edition's score down by **difficulty**. A cheap model might ace the easy questions and quietly fail the hard ones.

In [ ]:
by_level = pd.DataFrame(
    {name: lab.scores_by(r["per_item"], test_cases, "difficulty") for name, r in runs.items()}
)
by_level["v2"] = pd.Series(lab.scores_by(v2_row["per_item"], test_cases, "difficulty"))
by_level.loc[["easy", "medium", "hard"]]

**Think:** where does the cheapest edition lose the most? That's the heart of the router trade-off: **savings on easy questions, risk on hard ones.** If your customers mostly ask hard questions, the answer changes.

## 6.7 How do I decide without fooling myself?

Here's a discipline worth keeping: **decide the rule before looking at the winner.** Ours:

> Promote the edition that uses the **fewest tokens**, as long as its score is **within 0.1** of v2's.

If none qualify, we keep v2. That's a legitimate result too.

In [ ]:
TOLERANCE = 0.1  # how much quality we're willing to give up for savings

qualified = {
    name: r for name, r in runs.items()
    if r["score"] is not None and r["score"] >= v2_row["score"] - TOLERANCE
}

if qualified:
    winner = min(qualified, key=lambda name: qualified[name]["tokens"])
    saved = round(100 * (1 - qualified[winner]["tokens"] / v2_row["tokens"]))
    print(f"Winner: {winner}. Score {qualified[winner]['score']} vs v2 {v2_row['score']}, "
          f"using about {saved}% fewer tokens.")
else:
    winner = None
    print("No edition held quality within the tolerance. We keep v2. That's a real answer, too.")

**Expected:** one line naming a winner and its token savings, or a line saying no edition qualified.

**Think:** would you have picked a different winner by eye? Would a different tolerance (0.05 or 0.3) change it? Try it: edit `TOLERANCE` and re-run the cell above.

## 6.8 How do I make the winner official?

Promotion is just making the winner the newest version.

In [ ]:
if winner:
    v3 = project_client.agents.create_version(
        lab.AGENT_NAME,
        definition=PromptAgentDefinition(
            model=winner,
            instructions=v2.definition.instructions,
            tools=[FileSearchTool(vector_store_ids=[vector_store.id])],
        ),
    )
    print(f"v3 = Foundry version {v3.version}, running on {winner}")
    lab.save_state(v3_version=v3.version, v3_edition=winner)
else:
    print("Nothing to promote. v2 stays our best version.")

## 6.9 What does the whole climb look like?

Four versions, six ways. This table is the story of the lab: it started dark and lit up as we climbed. Green is good.

In [ ]:
scores = state["test_scores"]
picture = pd.DataFrame({
    "v0": lab.scores_by_way(scores["v0 on test"]["per_item"], test_cases),
    "v1": lab.scores_by_way(scores["v1 on test"]["per_item"], test_cases),
    "v2": lab.scores_by_way(scores["v2 on test"]["per_item"], test_cases),
})
if winner:
    picture["v3"] = lab.scores_by_way(runs[winner]["per_item"], test_cases)

try:
    display(picture.style.background_gradient(cmap="RdYlGn", vmin=1, vmax=5).format("{:.1f}"))
except Exception:
    print(picture)  # plain table if the colors aren't available

**Think:**
- Which row lit up the most between v0 and v2? That's the **quality** climb: instructions.
- Did v3 hold the line? That's the **cost** climb: the router.
- Which row is still the lowest? That's where our next experiment should go.

## 6.10 Is this the end?

A climb like this isn't one-and-done. Real agents get new questions every day. The same loop keeps going:

1. **Watch** production traffic with insights and continuous evaluation.
2. **Harvest** the questions that went wrong and add them to our test set as new, harder cases.
3. **Measure** again with the same rubric. A lower score on a harder set is a *good* sign: we found something new.
4. **Climb** again, one lever at a time.

**Never shrink the test set or loosen the rubric to bring a score back up.** Fix the agent instead.

## 6.11 What should I try next?

- **What if** we used `mr-cost` everywhere? Where does it break?
- **What if** we used `mr-subset`, a hand-picked list of models? Does it beat the default?
- **What if** we asked the agent optimizer to try **three router options** itself? (It can search over models, not just instructions: set `target_attributes` to include `model` and list the editions in `model_search_space`.)

## 6.12 Where did the savings actually come from? (the aha)

We started by asking whether we can cut the bill without losing quality. One more look, this time at *where* the winner differs from v2.

In [ ]:
# Winner minus v2, for each difficulty. Near zero means quality held; negative means it slipped.
if winner:
    gap = (by_level[winner] - by_level["v2"]).round(2)
    print(gap.loc[["easy", "medium", "hard"]])
else:
    print("No winner this time, so there's no gap to show.")

> **Aha:** The savings and the risk live in different places. Easy questions are where a router can be cheap and the score barely moves. Hard questions are where any quality loss shows up first. So "can I cut the bill?" really means "**how many of my questions are easy?**" Measure your own mix and you already know your answer.

We also took away:

- **Most questions are easy.** Paying frontier prices for all of them is the hidden cost.
- **A router trades savings against risk.** The risk lives in the hard questions.
- A good promotion rule is **written before looking at the winner**.
- **One lever at a time** kept every result explainable, across all three climbs.
- Insights showed us the slope, the rubric told us the height, the optimizer climbed on quality and the router climbed on cost.

**Go further (optional):** point the optimizer at the model lever. Set `target_attributes` to `["model"]` and list the four router editions in `model_search_space`. Does it pick the same edition we did?

**Nice work. We built it, measured it, broke it, fixed it and understood why.**